In [0]:
dbutils.library.restartPython()

In [0]:
%run ../setup_path

In [0]:
from pyspark.sql import functions as F
import yaml
from utils.data_transformation import apply_data_transformation

In [0]:
yaml_file = yaml.load(open("./mapping/silver_ocm_connections.yaml"), Loader=yaml.FullLoader)

PROJECT_ID = yaml_file['project_id']
SOURCE_TABLE = yaml_file['source_table']
TARGET_TABLE = yaml_file['target_table']
TARGET_SCHEMA = [c['target'] for c in yaml_file['schema']]

In [0]:
yaml_file['drop']

In [0]:
RUN_ID = 1793953962649

ocm_connections_df = (
                spark.table(f"`{PROJECT_ID}`.{SOURCE_TABLE}")
                .filter(F.col("_run_id") == RUN_ID)
)


In [0]:
ocm_connections_df = apply_data_transformation(ocm_connections_df, yaml_file)

In [0]:
spark.sql(f"""DELETE FROM `{PROJECT_ID}`.{TARGET_TABLE} WHERE RUN_ID = {RUN_ID}""")

In [0]:
ocm_connections_df = ocm_connections_df.select(TARGET_SCHEMA)
ocm_connections_df.write.mode("append").option("replaceWhere", f"RUN_ID = {RUN_ID}").saveAsTable("`dp-ev_charging-001`.silver.ocm_connections")